# Rice leaf disease: leakage-aware InceptionV4 study (full run)

This notebook reproduces every experiment of the study with the code in
<https://github.com/EgwuSamuel/rice-inceptionv4-repro>.

**Before running on Kaggle**
1. *Add Input*: dataset **bahribahri/riceleaf** and competition **paddy-disease-classification**
   (join the competition and accept its rules first).
2. *Settings*: Accelerator **GPU T4 x1** (or P100), Internet **on**.
3. *Save Version → Save & Run All (Commit)*. Expected time: about 6–9 GPU-hours.

**Outputs** (in `/kaggle/working`): `results_for_paper.zip` (tables, figures, logs, predictions)
and `app_bundle.zip` (ONNX model for the web app).

**Splitting over two sessions (optional).** In session 1 set `MODELS = "inception_v4,inception_v3,resnet50"`.
In session 2, add the output of session 1 as an input, set `PREVIOUS_RESULTS` to its `results` folder,
and set `MODELS = "efficientnet_b0,mobilenetv3_large,convnext_tiny,deit_small"`. Finished runs are skipped.

In [ ]:
import os, sys, shutil, subprocess, pathlib, importlib.util

# ---- options ---------------------------------------------------------------
SMOKE = os.environ.get("RICEDX_SMOKE", "0") == "1"   # local smoke test only
REPO_URL = "https://github.com/EgwuSamuel/rice-inceptionv4-repro.git"
MODELS = ""              # "" = all models; or a comma-separated subset
PREVIOUS_RESULTS = ""    # path to a results folder from an earlier session, to resume

In [ ]:
# ---- get the code ------------------------------------------------------------
here = pathlib.Path.cwd()
if (here.parent / "src" / "ricedx").is_dir():
    REPO = here.parent                       # notebook opened inside a local clone
elif (here / "src" / "ricedx").is_dir():
    REPO = here
else:
    base = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").is_dir() else here
    REPO = base / "rice-inceptionv4-repro"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

missing = [p for p in ("timm", "onnx", "onnxruntime", "yaml") if importlib.util.find_spec(p) is None]
if missing:
    pip_names = {"yaml": "pyyaml"}
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[pip_names.get(m, m) for m in missing]], check=True)

import torch, timm
print("repo:", REPO)
print("torch", torch.__version__, "| timm", timm.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# ---- configuration and output folder -----------------------------------------
os.chdir(REPO)
if SMOKE:
    subprocess.run([sys.executable, "scripts/make_synthetic_data.py", "--out", "smoke_data"], check=True)
    CONFIG = "configs/smoke.yaml"
    OUT = REPO / "smoke_results"
else:
    CONFIG = "configs/default.yaml"
    OUT = pathlib.Path("/kaggle/working/results") if pathlib.Path("/kaggle/working").is_dir() else REPO / "results"
OUT.mkdir(parents=True, exist_ok=True)
if PREVIOUS_RESULTS:
    shutil.copytree(PREVIOUS_RESULTS, OUT, dirs_exist_ok=True)
    print("resumed from", PREVIOUS_RESULTS)

from ricedx.run_all import main as run

def stage(*names):
    args = ["--config", CONFIG, "--out", str(OUT), "--stages", ",".join(names)]
    if MODELS:
        args += ["--models", MODELS]
    assert run(args) == 0
print("config:", CONFIG, "| output:", OUT)

## E1 — Leakage audit (image-level vs source-grouped split)

In [ ]:
stage("leakage")

## E2 + E3 — Source-grouped cross-validation of all models, with zero-shot external test

In [ ]:
stage("benchmark")

## E4 + E5 — Grad-CAM, ONNX export and quantisation, and the report

In [ ]:
stage("explain", "export", "report")

## Results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown((OUT / "summary.md").read_text(encoding="utf-8")))
for fig in sorted((OUT / "figures").glob("*.png")):
    print(fig.name)
    display(Image(filename=str(fig), width=520))

In [ ]:
# ---- package outputs -----------------------------------------------------------
dest = OUT.parent
light = dest / "results_for_paper"
if light.exists():
    shutil.rmtree(light)
shutil.copytree(OUT, light, ignore=shutil.ignore_patterns("*.pt", "*.onnx", "checkpoints", "export", "app_bundle"))
print(shutil.make_archive(str(light), "zip", light))
if (OUT / "app_bundle").exists():
    print(shutil.make_archive(str(dest / "app_bundle"), "zip", OUT / "app_bundle"))
shutil.rmtree(light)